In [31]:
import pandas

train_df   = pandas.read_csv("../../data/raw/hanwoo_train.csv")
death_df   = pandas.read_csv("../../data/raw/hanwoo_death.csv")

print("\n=== train_df ===")
print(train_df.shape)
print(train_df.columns.tolist())

print("\n=== death_df ===")
print(death_df.shape)
print(death_df.head())
print(death_df.columns.tolist())



=== train_df ===
(2408699, 23)
['sido', 'sigungu', 'eupmyeondong', 'stn', 'ABATT_DATE', 'JUDGE_DATE', 'JUDGE_SEX', 'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'WGRADE', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE', 'BIRTH_YMD', 'CATTLE_NO', 'FARM_UNIQUE_NO', 'LAST_GRADE']

=== death_df ===
(321389, 4)
             FARM_UNIQUE_NO  BIRTH_YMD  DEAD_YMD DEAD_REASON
0  WiJdwGdfb+CC7Jg0Pn/aLA==   20070621  20250107         -99
1  gj6CI5LHYXz7druWs1ZEUQ==   20060705  20250106         -99
2  We6zq4c9yoG5GKrt1CbrdA==   20080610  20250103         -99
3  s67fGrSPXhX1Bgp+JDW9Aw==   20050408  20250108         -99
4  3lGF4dGtWtydlceogBluUA==   20080425  20250109         -99
['FARM_UNIQUE_NO', 'BIRTH_YMD', 'DEAD_YMD', 'DEAD_REASON']


In [32]:
# train_df 날짜 변환
train_df["ABATT_DATE"] = pandas.to_datetime(train_df["ABATT_DATE"])
train_df["JUDGE_DATE"]  = pandas.to_datetime(train_df["JUDGE_DATE"])
train_df["BIRTH_YMD"]  = pandas.to_datetime(
    train_df["BIRTH_YMD"].astype(str), format="%Y%m%d", errors="coerce"
)

# death_df 날짜 변환 및 -99 처리
death_df["DEAD_YMD"]   = pandas.to_datetime(
    death_df["DEAD_YMD"].astype(str), format="%Y%m%d", errors="coerce"
)
death_df["BIRTH_YMD"]  = pandas.to_datetime(
    death_df["BIRTH_YMD"].astype(str), format="%Y%m%d", errors="coerce"
)
# DEAD_REASON은 문자열이므로 문자열 "-99"로 처리
death_df["DEAD_REASON"] = death_df["DEAD_REASON"].astype(str).replace("-99", None)

print("=== train_df 날짜 변환 확인 ===")
print(train_df[["ABATT_DATE", "JUDGE_DATE", "BIRTH_YMD"]].dtypes)
print(train_df[["ABATT_DATE", "JUDGE_DATE", "BIRTH_YMD"]].head())

print("\n=== death_df 날짜 변환 확인 ===")
print(death_df[["DEAD_YMD", "BIRTH_YMD"]].dtypes)
print(death_df.head())
print(f"\nDEAD_REASON 결측 수: {death_df['DEAD_REASON'].isnull().sum():,}")

=== train_df 날짜 변환 확인 ===
ABATT_DATE    datetime64[us]
JUDGE_DATE    datetime64[us]
BIRTH_YMD     datetime64[us]
dtype: object
  ABATT_DATE JUDGE_DATE  BIRTH_YMD
0 2023-01-01 2023-01-02 2016-09-15
1 2023-01-01 2023-01-02 2020-05-04
2 2023-01-01 2023-01-02 2020-08-03
3 2023-01-01 2023-01-02 2020-05-10
4 2023-01-01 2023-01-02 2020-05-11

=== death_df 날짜 변환 확인 ===
DEAD_YMD     datetime64[us]
BIRTH_YMD    datetime64[us]
dtype: object
             FARM_UNIQUE_NO  BIRTH_YMD   DEAD_YMD DEAD_REASON
0  WiJdwGdfb+CC7Jg0Pn/aLA== 2007-06-21 2025-01-07         NaN
1  gj6CI5LHYXz7druWs1ZEUQ== 2006-07-05 2025-01-06         NaN
2  We6zq4c9yoG5GKrt1CbrdA== 2008-06-10 2025-01-03         NaN
3  s67fGrSPXhX1Bgp+JDW9Aw== 2005-04-08 2025-01-08         NaN
4  3lGF4dGtWtydlceogBluUA== 2008-04-25 2025-01-09         NaN

DEAD_REASON 결측 수: 234,276


In [33]:
# FARM_UNIQUE_NO 기준 death 집계 (death_count만)
death_summary = death_df.groupby("FARM_UNIQUE_NO").agg(
    death_count=("DEAD_YMD", "count")
).reset_index()

print("=== death_summary 확인 ===")
print(f"행 수: {len(death_summary):,}")
print(death_summary.head())

# train_df에 병합
before_rows = len(train_df)
before_cols = len(train_df.columns)

train_df = train_df.merge(
    death_summary,
    on="FARM_UNIQUE_NO",
    how="left"
)

after_rows = len(train_df)
after_cols = len(train_df.columns)

print(f"\n=== 병합 결과 ===")
print(f"병합 전: {before_rows:,}행 {before_cols}열")
print(f"병합 후: {after_rows:,}행 {after_cols}열")
print(f"\ndeath_count 결측: {train_df['death_count'].isnull().sum():,}")

# 저장
import os
os.makedirs("../../data/processed", exist_ok=True)
train_df.to_csv("../data/processed/step1_death.csv", index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {train_df.shape}")

=== death_summary 확인 ===
행 수: 50,623
             FARM_UNIQUE_NO  death_count
0  ++3dDT9h9GLLBVoQDgn1IA==            4
1  ++EgHgT0FBhXP2UY4tsA2Q==            2
2  ++JGJuguC4RUqLsXvUOLmQ==            1
3  ++RjpVT+dYlDXbsNj/zxTg==           86
4  ++UxtqCtObZiLZCTuRaAyQ==           17

=== 병합 결과 ===
병합 전: 2,408,699행 23열
병합 후: 2,408,699행 24열

death_count 결측: 518,737

저장 완료: (2408699, 24)
